# Notebook 01: DIR-Lab Deformable Registration, TRE & Parametric Response Mapping

This notebook walks through:
1. Loading paired inspiratory and expiratory thoracic CT scans from the DIR-Lab dataset.
2. Morphological lung parenchyma segmentation.
3. Deformable BSpline registration using SimpleITK.
4. Evaluating Target Registration Error (TRE) in mm on 300 anatomical landmarks.
5. Quantifying Parametric Response Mapping (PRM) into normal, fSAD, and emphysema phenotypes.


In [ ]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from pathlib import Path

from copd_anomaly.registration.registrar import SimpleITKRegistrar, evaluate_tre
from copd_anomaly.features.prm import compute_prm_map, compute_prm_stats
from copd_anomaly.segmentation.segmenter import LungSegmenter
from copd_anomaly.data.dirlab_loader import DIRLabLoader

print('Loaded registration and PRM modules successfully.')


## 1. Inspecting the Generated DIR-Lab Manifest
All 20 scans across the 10 DIR-Lab cases have verified Hounsfield Unit offsets and 0-based landmark coordinate alignment.


In [ ]:
manifest_path = Path('../data/processed/dirlab_manifest.parquet')
if manifest_path.exists():
    df_manifest = pd.read_parquet(manifest_path)
    display(df_manifest[['scan_id', 'hu_offset_verified', 'lung_median_hu', 'landmark_index_base', 'landmark_in_lung_frac']])
else:
    print('Manifest file not found on local disk. Run scripts/build_dirlab_manifest.py first.')


## 2. Interactive Synthetic Demonstration: Registration & TRE Evaluation
We simulate a 3D paired volume with known ground-truth landmark displacement to demonstrate TRE calculation and PRM classification.


In [ ]:
# Create synthetic pair phantom
shape = (20, 32, 32)
vol_insp = np.full(shape, 40.0, dtype=np.float32)
vol_insp[3:17, 8:24, 6:26] = -850.0  # Lung parenchyma

vol_exp = np.full(shape, 40.0, dtype=np.float32)
vol_exp[3:17, 8:24, 7:27] = -800.0   # Exhale shifted by 1 voxel in X

segmenter = LungSegmenter(backend='threshold')
mask = segmenter.segment(vol_insp, spacing_zyx=(2.0, 1.0, 1.0))

registrar = SimpleITKRegistrar(fast_mode=True)
reg_res = registrar.register(vol_insp, vol_exp, fixed_mask=mask, spacing_zyx=(2.0, 1.0, 1.0))

print(f'Registration complete. Warped volume shape: {reg_res.warped_volume.shape}')


In [ ]:
# Evaluate Target Registration Error (TRE)
fixed_landmarks = np.array([[10.0, 16.0, 12.0], [12.0, 18.0, 20.0]], dtype=np.float32)
moving_landmarks = fixed_landmarks + np.array([2.0, 2.0, 0.0], dtype=np.float32)

warped_landmarks = registrar.transform_points(moving_landmarks, reg_res.transform, spacing_zyx=(2.0, 1.0, 1.0))
tre_stats = evaluate_tre(fixed_landmarks, moving_landmarks, warped_landmarks, spacing_zyx=(2.0, 1.0, 1.0))

print('TRE Statistics (mm):')
for k, v in tre_stats.items():
    print(f'  {k}: {v:.2f}')


## 3. Parametric Response Mapping (PRM)
Classifies lung voxels into Normal, Functional Small Airway Disease (fSAD), and Emphysema.


In [ ]:
prm_stats = compute_prm_stats(vol_insp, reg_res.warped_volume, mask, spacing_zyx=(2.0, 1.0, 1.0))
print('PRM Quantitative Breakdown:')
for k, v in prm_stats.items():
    print(f'  {k}: {v:.2f}%')
